# Null Catamenial Epilepsy Analysis Notebook

Populated from `outputs_bench_200` outputs. Detected analysis mode: **smoke**.

- Participants: **200** (healthy ovulatory: 100, heterogeneous menstruating-age: 100)
- Primary window rows: **5,200**
- Study-level Monte Carlo rows: **27,600**
- Manifest files: **25**

## Cohort terminology

This notebook uses **heterogeneous menstruating-age** as the presentation label for the broader cohort key stored in the analysis files. In this null-simulation study it means an assumption-driven broader menstruating-age simulated cohort, not a disease-positive, clinically diagnosed, or demographically representative population. This cohort allows the hormone-cycle simulator's natural ovulatory and anovulatory behavior and its configured rates of cycle modifiers such as PCOS, peri-menarche, perimenopause, dysmenorrhea, and cycle irregularity when available. It is contrasted with the **healthy ovulatory** cohort, which is restricted to adult ovulatory cycling with those medical modifiers disabled where the simulator exposes those controls. In both cohorts, seizure diaries are generated independently from menstrual diaries using separate deterministic random streams, so any apparent catamenial epilepsy classification is a false positive under the null.

## Exact analysis plan followed

1. Simulate two cohorts separately and never pool results: healthy ovulatory and heterogeneous menstruating-age.
2. Full defined cohort sizes are `{'healthy ovulatory': 50000, 'heterogeneous menstruating-age': 50000}`; smoke mode uses `100` total participants.
3. For each participant, simulate an independent CHOCOLATES seizure diary and an independent hormone-cycle diary for `36` months in full mode.
4. Align the independently generated seizure and hormone diaries directly by calendar-day index without reordering.
5. Label phases on the full diary before subsetting windows, using strict Herzog labels for primary analyses and a luteal-anchored fixed ovulatory window for sensitivity analyses.
6. Sample calendar windows, full 36-month windows, and complete-cycle windows exactly as configured.
7. Classify windows using exact Herzog 2004, windowed Herzog thresholds, C3-exclusion and pattern-only sensitivities, minimum-data rules, reproducibility rules, full-window stabilized/window-dispersion NB regression, and assumption-based historical definitions.
8. Summarize false positives and indeterminacy by cohort, phase mode, window, definition, seizure-burden stratum, participant-level status, pattern category, and study-level Monte Carlo benchmarks.
9. Save outputs as parquet/CSV, publication figures as PNG/PDF/SVG, a 1% daily audit sample, and a manifest.

### Recorded assumptions

- Definition D uses a participant-full-diary method-of-moments negative-binomial alpha recorded in d_alpha; Poisson robust fallback is recorded in d_reason when statsmodels NB fitting fails. Definition D_window_alpha re-estimates alpha from the analyzed window as a non-oracle sensitivity.
- Healthy ovulatory cohort used hormone_cycler build_patient_profile/render_cycle with ovulation_probability set to 1.0 because simulate_diary does not expose a public force-ovulation knob.
- Historical definitions H1-H4 are assumption-based operationalizations and are flagged in summary outputs.
- Study-level Monte Carlo samples each selected participant from a deterministic pool of precomputed random valid 3-month windows to avoid retaining all daily diaries in memory.
- The hormone simulator exposes medical-factor knobs but no natural prevalence sampler; heterogeneous menstruating-age medical factors were sampled from config.yaml rates.

## Reproducible function calls

These cells are the exact calls used to regenerate the analysis outputs. Run the smoke call for a quick end-to-end check; run the full call for the defined 100,000-participant analysis.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from paper1_null_ce.core.utils import load_config
from paper1_null_ce.core.simulate import run_pipeline

config = load_config(ROOT / "config.yaml")

# Quick validation run used while developing and reviewing the pipeline:
smoke_result = run_pipeline(config, mode="smoke")

# Prespecified full analysis. This is intentionally separate because it is large:
# full_result = run_pipeline(config, mode="full")

# During a long run, check ETA from another terminal:
# python3.11 scripts/check_paper1_progress.py --progress outputs/progress.json


## Load the current populated outputs

The remaining notebook cells read the existing output artifacts. This keeps figure and table rendering fast and reproducible after either a smoke run or a full run.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = ROOT / "outputs"

participant_summary = pd.read_parquet(OUTPUT_DIR / "participant_summary.parquet")
window_results = pd.read_parquet(OUTPUT_DIR / "window_results.parquet")
study_path = OUTPUT_DIR / "study_level_3month.parquet"
if not study_path.exists():
    study_path = OUTPUT_DIR / "study_level_3month_n30.parquet"
study_level = pd.read_parquet(study_path)
summary_tables = pd.read_csv(OUTPUT_DIR / "summary_tables.csv")
manifest = json.loads((OUTPUT_DIR / "manifest.json").read_text())

participant_summary.shape, window_results.shape, study_level.shape, summary_tables.shape


## Table 1. Cohort and diary summary

**Why this table is included.** This table verifies that both defined cohorts are represented separately, that cycle summaries are available, and that seizure-burden metrics were carried through from the seizure simulator. It is the first QC table because every downstream apparent-classification estimate depends on the cohort construction and diary burden.

**Code to call.**

In [ ]:
cohort_summary = (
    participant_summary
    .groupby("cohort")
    .agg(
        participants=("participant_id", "nunique"),
        age_mean=("age", "mean"),
        age_sd=("age", "std"),
        mean_cycle_length=("mean_cycle_length", "mean"),
        sd_cycle_length=("sd_cycle_length", "mean"),
        ovulatory_fraction=("ovulatory_fraction", "mean"),
        seizure_days_per_month=("seizure_days_per_month", "mean"),
        seizures_per_month=("seizures_per_month", "mean"),
    )
    .reset_index()
)
cohort_summary


| Cohort                         | Participants | Mean age, years | Age SD, years | Mean cycle length, days | Mean cycle-length SD, days | Ovulatory cycles | Seizure days per month | Seizures per month |
| ------------------------------ | ------------ | --------------- | ------------- | ----------------------- | -------------------------- | ---------------- | ---------------------- | ------------------ |
| healthy ovulatory              | 100          | 32.9            | 7.4           | 28.83                   | 3.26                       | 100.0%           | 2.28                   | 5.71               |
| heterogeneous menstruating-age | 100          | 35.1            | 12.8          | 31.47                   | 5.33                       | 77.4%            | 2.46                   | 6.74               |

**Table 1 caption.** Cohort-level participant and diary summaries for the full simulation. Percentages use a 0-100% scale; seizure rates are monthly averages over the 36-month diary.

## Table 2. Primary full-window false-positive rates

**Why this table is included.** This table is the primary result summary for person-window false-positive rates under the null. It uses the full diary window and reports classifiable denominators, positives, Wilson 95% intervals, and indeterminate rates separately by cohort and definition.

**Code to call.**

In [ ]:
primary_full = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin([
        "A_windowed_any", "A_windowed_C1_or_C2",
        "B_minimum_data_any", "B_minimum_data_C1_or_C2",
        "C_reproducibility_any", "D_nb_regression_C1_or_C2"
    ]))
].copy()
primary_full


| Cohort                         | CE definition                           | Windows analyzed | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | --------------------------------------- | ---------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Windowed Herzog C1/C2 union             | 100              | 98                   | 16                     | 16.3% (10.3, 24.9)           | 2.0%                  |
| healthy ovulatory              | Windowed Herzog thresholds              | 100              | 98                   | 16                     | 16.3% (10.3, 24.9)           | 2.0%                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data | 100              | 90                   | 12                     | 13.3% (7.8, 21.9)            | 10.0%                 |
| healthy ovulatory              | Windowed Herzog with minimum data       | 100              | 90                   | 12                     | 13.3% (7.8, 21.9)            | 10.0%                 |
| healthy ovulatory              | Cycle reproducibility, 6-cycle rule     | 100              | 9                    | 0                      | 0.0% (0.0, 29.9)             | 91.0%                 |
| healthy ovulatory              | Negative-binomial regression C1/C2      | 100              | 90                   | 5                      | 5.6% (2.4, 12.4)             | 10.0%                 |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 union             | 100              | 99                   | 11                     | 11.1% (6.3, 18.8)            | 1.0%                  |
| heterogeneous menstruating-age | Windowed Herzog thresholds              | 100              | 99                   | 42                     | 42.4% (33.2, 52.3)           | 1.0%                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data | 100              | 97                   | 10                     | 10.3% (5.7, 17.9)            | 3.0%                  |
| heterogeneous menstruating-age | Windowed Herzog with minimum data       | 100              | 97                   | 41                     | 42.3% (32.9, 52.2)           | 3.0%                  |
| heterogeneous menstruating-age | Cycle reproducibility, 6-cycle rule     | 100              | 26                   | 8                      | 30.8% (16.5, 50.0)           | 74.0%                 |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2      | 100              | 97                   | 7                      | 7.2% (3.5, 14.2)             | 3.0%                  |

**Table 2 caption.** Primary full-diary false-positive rates under the null. The denominator for the false-positive rate is the number of classifiable participant windows, and the confidence interval is Wilson 95%.

## Table 3. Window-length sensitivity for core definitions

**Why this table is included.** This table shows why diary length matters. Short calendar windows can be classifiable for simple windowed ratios but not for minimum-data, reproducibility, or exact three-cycle rules; the indeterminate column quantifies that tradeoff.

**Code to call.**

In [ ]:
window_sensitivity = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.definition.isin([
        "A_exact_any", "A_windowed_any", "A_windowed_C1_or_C2",
        "B_minimum_data_C1_or_C2", "C_reproducibility_C1_or_C2",
        "D_nb_regression_C1_or_C2"
    ]))
].copy()
window_sensitivity


| Cohort                         | Observation window  | CE definition                             | Classifiable windows | False-positive windows | False-positive rate | Indeterminate windows |
| ------------------------------ | ------------------- | ----------------------------------------- | -------------------- | ---------------------- | ------------------- | --------------------- |
| healthy ovulatory              | 1 month             | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Windowed Herzog C1/C2 union               | 73                   | 33                     | 45.2%               | 27.0%                 |
| healthy ovulatory              | 1 month             | Windowed Herzog thresholds                | 73                   | 33                     | 45.2%               | 27.0%                 |
| healthy ovulatory              | 1 month             | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 1 month             | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 cycles           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 cycles           | Windowed Herzog C1/C2 union               | 91                   | 22                     | 24.2%               | 9.0%                  |
| healthy ovulatory              | 12 cycles           | Windowed Herzog thresholds                | 91                   | 22                     | 24.2%               | 9.0%                  |
| healthy ovulatory              | 12 cycles           | Windowed Herzog C1/C2 with minimum data   | 84                   | 19                     | 22.6%               | 16.0%                 |
| healthy ovulatory              | 12 cycles           | Cycle reproducibility C1/C2, 6-cycle rule | 27                   | 0                      | 0.0%                | 73.0%                 |
| healthy ovulatory              | 12 cycles           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 12 months           | Windowed Herzog C1/C2 union               | 91                   | 25                     | 27.5%               | 9.0%                  |
| healthy ovulatory              | 12 months           | Windowed Herzog thresholds                | 91                   | 25                     | 27.5%               | 9.0%                  |
| healthy ovulatory              | 12 months           | Windowed Herzog C1/C2 with minimum data   | 84                   | 22                     | 26.2%               | 16.0%                 |
| healthy ovulatory              | 12 months           | Cycle reproducibility C1/C2, 6-cycle rule | 29                   | 0                      | 0.0%                | 71.0%                 |
| healthy ovulatory              | 12 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 18 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 18 months           | Windowed Herzog C1/C2 union               | 96                   | 18                     | 18.8%               | 4.0%                  |
| healthy ovulatory              | 18 months           | Windowed Herzog thresholds                | 96                   | 18                     | 18.8%               | 4.0%                  |
| healthy ovulatory              | 18 months           | Windowed Herzog C1/C2 with minimum data   | 86                   | 15                     | 17.4%               | 14.0%                 |
| healthy ovulatory              | 18 months           | Cycle reproducibility C1/C2, 6-cycle rule | 19                   | 0                      | 0.0%                | 81.0%                 |
| healthy ovulatory              | 18 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 24 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 24 months           | Windowed Herzog C1/C2 union               | 95                   | 20                     | 21.1%               | 5.0%                  |
| healthy ovulatory              | 24 months           | Windowed Herzog thresholds                | 95                   | 20                     | 21.1%               | 5.0%                  |
| healthy ovulatory              | 24 months           | Windowed Herzog C1/C2 with minimum data   | 86                   | 15                     | 17.4%               | 14.0%                 |
| healthy ovulatory              | 24 months           | Cycle reproducibility C1/C2, 6-cycle rule | 14                   | 0                      | 0.0%                | 86.0%                 |
| healthy ovulatory              | 24 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 cycles            | Exact Herzog 2004, any CE pattern         | 41                   | 20                     | 48.8%               | 59.0%                 |
| healthy ovulatory              | 3 cycles            | Windowed Herzog C1/C2 union               | 86                   | 39                     | 45.3%               | 14.0%                 |
| healthy ovulatory              | 3 cycles            | Windowed Herzog thresholds                | 86                   | 39                     | 45.3%               | 14.0%                 |
| healthy ovulatory              | 3 cycles            | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Windowed Herzog C1/C2 union               | 89                   | 35                     | 39.3%               | 11.0%                 |
| healthy ovulatory              | 3 months            | Windowed Herzog thresholds                | 89                   | 35                     | 39.3%               | 11.0%                 |
| healthy ovulatory              | 3 months            | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 3 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36 months           | Windowed Herzog C1/C2 union               | 98                   | 16                     | 16.3%               | 2.0%                  |
| healthy ovulatory              | 36 months           | Windowed Herzog thresholds                | 98                   | 16                     | 16.3%               | 2.0%                  |
| healthy ovulatory              | 36 months           | Windowed Herzog C1/C2 with minimum data   | 90                   | 12                     | 13.3%               | 10.0%                 |
| healthy ovulatory              | 36 months           | Cycle reproducibility C1/C2, 6-cycle rule | 9                    | 0                      | 0.0%                | 91.0%                 |
| healthy ovulatory              | 36 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36-month full diary | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 36-month full diary | Windowed Herzog C1/C2 union               | 98                   | 16                     | 16.3%               | 2.0%                  |
| healthy ovulatory              | 36-month full diary | Windowed Herzog thresholds                | 98                   | 16                     | 16.3%               | 2.0%                  |
| healthy ovulatory              | 36-month full diary | Windowed Herzog C1/C2 with minimum data   | 90                   | 12                     | 13.3%               | 10.0%                 |
| healthy ovulatory              | 36-month full diary | Cycle reproducibility C1/C2, 6-cycle rule | 9                    | 0                      | 0.0%                | 91.0%                 |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression C1/C2        | 90                   | 5                      | 5.6%                | 10.0%                 |
| healthy ovulatory              | 4 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 4 months            | Windowed Herzog C1/C2 union               | 89                   | 30                     | 33.7%               | 11.0%                 |
| healthy ovulatory              | 4 months            | Windowed Herzog thresholds                | 89                   | 30                     | 33.7%               | 11.0%                 |
| healthy ovulatory              | 4 months            | Windowed Herzog C1/C2 with minimum data   | 70                   | 21                     | 30.0%               | 30.0%                 |
| healthy ovulatory              | 4 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 4 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 cycles            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 cycles            | Windowed Herzog C1/C2 union               | 93                   | 37                     | 39.8%               | 7.0%                  |
| healthy ovulatory              | 6 cycles            | Windowed Herzog thresholds                | 93                   | 37                     | 39.8%               | 7.0%                  |
| healthy ovulatory              | 6 cycles            | Windowed Herzog C1/C2 with minimum data   | 77                   | 29                     | 37.7%               | 23.0%                 |
| healthy ovulatory              | 6 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 46                   | 6                      | 13.0%               | 54.0%                 |
| healthy ovulatory              | 6 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 6 months            | Windowed Herzog C1/C2 union               | 90                   | 32                     | 35.6%               | 10.0%                 |
| healthy ovulatory              | 6 months            | Windowed Herzog thresholds                | 90                   | 32                     | 35.6%               | 10.0%                 |
| healthy ovulatory              | 6 months            | Windowed Herzog C1/C2 with minimum data   | 80                   | 30                     | 37.5%               | 20.0%                 |
| healthy ovulatory              | 6 months            | Cycle reproducibility C1/C2, 6-cycle rule | 20                   | 4                      | 20.0%               | 80.0%                 |
| healthy ovulatory              | 6 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 9 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | 9 months            | Windowed Herzog C1/C2 union               | 92                   | 21                     | 22.8%               | 8.0%                  |
| healthy ovulatory              | 9 months            | Windowed Herzog thresholds                | 92                   | 21                     | 22.8%               | 8.0%                  |
| healthy ovulatory              | 9 months            | Windowed Herzog C1/C2 with minimum data   | 81                   | 19                     | 23.5%               | 19.0%                 |
| healthy ovulatory              | 9 months            | Cycle reproducibility C1/C2, 6-cycle rule | 31                   | 1                      | 3.2%                | 69.0%                 |
| healthy ovulatory              | 9 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Windowed Herzog C1/C2 union               | 82                   | 38                     | 46.3%               | 18.0%                 |
| heterogeneous menstruating-age | 1 month             | Windowed Herzog thresholds                | 82                   | 39                     | 47.6%               | 18.0%                 |
| heterogeneous menstruating-age | 1 month             | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 1 month             | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 cycles           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 cycles           | Windowed Herzog C1/C2 union               | 99                   | 32                     | 32.3%               | 1.0%                  |
| heterogeneous menstruating-age | 12 cycles           | Windowed Herzog thresholds                | 99                   | 44                     | 44.4%               | 1.0%                  |
| heterogeneous menstruating-age | 12 cycles           | Windowed Herzog C1/C2 with minimum data   | 96                   | 29                     | 30.2%               | 4.0%                  |
| heterogeneous menstruating-age | 12 cycles           | Cycle reproducibility C1/C2, 6-cycle rule | 32                   | 0                      | 0.0%                | 68.0%                 |
| heterogeneous menstruating-age | 12 cycles           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 12 months           | Windowed Herzog C1/C2 union               | 98                   | 30                     | 30.6%               | 2.0%                  |
| heterogeneous menstruating-age | 12 months           | Windowed Herzog thresholds                | 98                   | 50                     | 51.0%               | 2.0%                  |
| heterogeneous menstruating-age | 12 months           | Windowed Herzog C1/C2 with minimum data   | 94                   | 26                     | 27.7%               | 6.0%                  |
| heterogeneous menstruating-age | 12 months           | Cycle reproducibility C1/C2, 6-cycle rule | 36                   | 0                      | 0.0%                | 64.0%                 |
| heterogeneous menstruating-age | 12 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 18 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 18 months           | Windowed Herzog C1/C2 union               | 99                   | 26                     | 26.3%               | 1.0%                  |
| heterogeneous menstruating-age | 18 months           | Windowed Herzog thresholds                | 99                   | 47                     | 47.5%               | 1.0%                  |
| heterogeneous menstruating-age | 18 months           | Windowed Herzog C1/C2 with minimum data   | 96                   | 24                     | 25.0%               | 4.0%                  |
| heterogeneous menstruating-age | 18 months           | Cycle reproducibility C1/C2, 6-cycle rule | 26                   | 0                      | 0.0%                | 74.0%                 |
| heterogeneous menstruating-age | 18 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 24 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 24 months           | Windowed Herzog C1/C2 union               | 99                   | 17                     | 17.2%               | 1.0%                  |
| heterogeneous menstruating-age | 24 months           | Windowed Herzog thresholds                | 99                   | 41                     | 41.4%               | 1.0%                  |
| heterogeneous menstruating-age | 24 months           | Windowed Herzog C1/C2 with minimum data   | 97                   | 16                     | 16.5%               | 3.0%                  |
| heterogeneous menstruating-age | 24 months           | Cycle reproducibility C1/C2, 6-cycle rule | 22                   | 0                      | 0.0%                | 78.0%                 |
| heterogeneous menstruating-age | 24 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 cycles            | Exact Herzog 2004, any CE pattern         | 37                   | 18                     | 48.6%               | 63.0%                 |
| heterogeneous menstruating-age | 3 cycles            | Windowed Herzog C1/C2 union               | 94                   | 38                     | 40.4%               | 6.0%                  |
| heterogeneous menstruating-age | 3 cycles            | Windowed Herzog thresholds                | 94                   | 46                     | 48.9%               | 6.0%                  |
| heterogeneous menstruating-age | 3 cycles            | Windowed Herzog C1/C2 with minimum data   | 4                    | 1                      | 25.0%               | 96.0%                 |
| heterogeneous menstruating-age | 3 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Windowed Herzog C1/C2 union               | 93                   | 38                     | 40.9%               | 7.0%                  |
| heterogeneous menstruating-age | 3 months            | Windowed Herzog thresholds                | 93                   | 43                     | 46.2%               | 7.0%                  |
| heterogeneous menstruating-age | 3 months            | Windowed Herzog C1/C2 with minimum data   | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 3 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36 months           | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36 months           | Windowed Herzog C1/C2 union               | 99                   | 11                     | 11.1%               | 1.0%                  |
| heterogeneous menstruating-age | 36 months           | Windowed Herzog thresholds                | 99                   | 42                     | 42.4%               | 1.0%                  |
| heterogeneous menstruating-age | 36 months           | Windowed Herzog C1/C2 with minimum data   | 97                   | 10                     | 10.3%               | 3.0%                  |
| heterogeneous menstruating-age | 36 months           | Cycle reproducibility C1/C2, 6-cycle rule | 13                   | 0                      | 0.0%                | 87.0%                 |
| heterogeneous menstruating-age | 36 months           | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36-month full diary | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 36-month full diary | Windowed Herzog C1/C2 union               | 99                   | 11                     | 11.1%               | 1.0%                  |
| heterogeneous menstruating-age | 36-month full diary | Windowed Herzog thresholds                | 99                   | 42                     | 42.4%               | 1.0%                  |
| heterogeneous menstruating-age | 36-month full diary | Windowed Herzog C1/C2 with minimum data   | 97                   | 10                     | 10.3%               | 3.0%                  |
| heterogeneous menstruating-age | 36-month full diary | Cycle reproducibility C1/C2, 6-cycle rule | 13                   | 0                      | 0.0%                | 87.0%                 |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression C1/C2        | 97                   | 7                      | 7.2%                | 3.0%                  |
| heterogeneous menstruating-age | 4 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 4 months            | Windowed Herzog C1/C2 union               | 94                   | 41                     | 43.6%               | 6.0%                  |
| heterogeneous menstruating-age | 4 months            | Windowed Herzog thresholds                | 94                   | 52                     | 55.3%               | 6.0%                  |
| heterogeneous menstruating-age | 4 months            | Windowed Herzog C1/C2 with minimum data   | 80                   | 33                     | 41.2%               | 20.0%                 |
| heterogeneous menstruating-age | 4 months            | Cycle reproducibility C1/C2, 6-cycle rule | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 4 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 cycles            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 cycles            | Windowed Herzog C1/C2 union               | 98                   | 28                     | 28.6%               | 2.0%                  |
| heterogeneous menstruating-age | 6 cycles            | Windowed Herzog thresholds                | 98                   | 45                     | 45.9%               | 2.0%                  |
| heterogeneous menstruating-age | 6 cycles            | Windowed Herzog C1/C2 with minimum data   | 87                   | 24                     | 27.6%               | 13.0%                 |
| heterogeneous menstruating-age | 6 cycles            | Cycle reproducibility C1/C2, 6-cycle rule | 55                   | 3                      | 5.5%                | 45.0%                 |
| heterogeneous menstruating-age | 6 cycles            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 6 months            | Windowed Herzog C1/C2 union               | 96                   | 30                     | 31.2%               | 4.0%                  |
| heterogeneous menstruating-age | 6 months            | Windowed Herzog thresholds                | 96                   | 46                     | 47.9%               | 4.0%                  |
| heterogeneous menstruating-age | 6 months            | Windowed Herzog C1/C2 with minimum data   | 85                   | 26                     | 30.6%               | 15.0%                 |
| heterogeneous menstruating-age | 6 months            | Cycle reproducibility C1/C2, 6-cycle rule | 8                    | 1                      | 12.5%               | 92.0%                 |
| heterogeneous menstruating-age | 6 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 9 months            | Exact Herzog 2004, any CE pattern         | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | 9 months            | Windowed Herzog C1/C2 union               | 97                   | 31                     | 32.0%               | 3.0%                  |
| heterogeneous menstruating-age | 9 months            | Windowed Herzog thresholds                | 97                   | 49                     | 50.5%               | 3.0%                  |
| heterogeneous menstruating-age | 9 months            | Windowed Herzog C1/C2 with minimum data   | 93                   | 28                     | 30.1%               | 7.0%                  |
| heterogeneous menstruating-age | 9 months            | Cycle reproducibility C1/C2, 6-cycle rule | 45                   | 1                      | 2.2%                | 55.0%                 |
| heterogeneous menstruating-age | 9 months            | Negative-binomial regression C1/C2        | 0                    | 0                      | NA                  | 100.0%                |

**Table 3 caption.** False-positive and indeterminate rates for every prespecified observation window and core definition. Exact Herzog 2004 is expected to be classifiable only for 3-complete-cycle windows.

## Table 4. Strict Herzog versus luteal-anchored ovulatory sensitivity

**Why this table is included.** This table addresses whether false-positive rates depend on the strict Herzog periovulatory window expanding with cycle length. Strict Herzog remains primary for historical comparability; the luteal-anchored mode fixes the ovulatory window at four pre-luteal days.

**Code to call.**

In [ ]:
phase_mode_sensitivity = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.subset == "all")
    & (
        (summary_tables.window_type == "full")
        | ((summary_tables.window_type == "calendar") & (summary_tables.window_value.astype(str) == "3"))
    )
    & (summary_tables.definition.isin(["A_windowed_any", "A_windowed_C1_or_C2", "A_windowed_C3_only"]))
].copy()
phase_mode_sensitivity


| Cohort                         | Phase labeling            | Observation window  | CE definition               | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | ------------------------- | ------------------- | --------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C1/C2 union | 89                   | 31                     | 34.8% (25.7, 45.2)           | 11.0%                 |
| healthy ovulatory              | Strict Herzog             | 3 months            | Windowed Herzog C1/C2 union | 89                   | 35                     | 39.3% (29.8, 49.7)           | 11.0%                 |
| healthy ovulatory              | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Strict Herzog             | 3 months            | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Luteal-anchored ovulatory | 3 months            | Windowed Herzog thresholds  | 89                   | 31                     | 34.8% (25.7, 45.2)           | 11.0%                 |
| healthy ovulatory              | Strict Herzog             | 3 months            | Windowed Herzog thresholds  | 89                   | 35                     | 39.3% (29.8, 49.7)           | 11.0%                 |
| healthy ovulatory              | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C1/C2 union | 98                   | 13                     | 13.3% (7.9, 21.4)            | 2.0%                  |
| healthy ovulatory              | Strict Herzog             | 36-month full diary | Windowed Herzog C1/C2 union | 98                   | 16                     | 16.3% (10.3, 24.9)           | 2.0%                  |
| healthy ovulatory              | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Strict Herzog             | 36-month full diary | Windowed Herzog C3 only     | 0                    | 0                      | NA                           | 100.0%                |
| healthy ovulatory              | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog thresholds  | 98                   | 13                     | 13.3% (7.9, 21.4)            | 2.0%                  |
| healthy ovulatory              | Strict Herzog             | 36-month full diary | Windowed Herzog thresholds  | 98                   | 16                     | 16.3% (10.3, 24.9)           | 2.0%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C1/C2 union | 93                   | 33                     | 35.5% (26.5, 45.6)           | 7.0%                  |
| heterogeneous menstruating-age | Strict Herzog             | 3 months            | Windowed Herzog C1/C2 union | 93                   | 38                     | 40.9% (31.4, 51.0)           | 7.0%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 3 months            | Windowed Herzog C3 only     | 34                   | 13                     | 38.2% (23.9, 55.0)           | 66.0%                 |
| heterogeneous menstruating-age | Strict Herzog             | 3 months            | Windowed Herzog C3 only     | 33                   | 17                     | 51.5% (35.2, 67.5)           | 67.0%                 |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 3 months            | Windowed Herzog thresholds  | 93                   | 41                     | 44.1% (34.4, 54.2)           | 7.0%                  |
| heterogeneous menstruating-age | Strict Herzog             | 3 months            | Windowed Herzog thresholds  | 93                   | 43                     | 46.2% (36.5, 56.3)           | 7.0%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C1/C2 union | 99                   | 12                     | 12.1% (7.1, 20.0)            | 1.0%                  |
| heterogeneous menstruating-age | Strict Herzog             | 36-month full diary | Windowed Herzog C1/C2 union | 99                   | 11                     | 11.1% (6.3, 18.8)            | 1.0%                  |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog C3 only     | 84                   | 27                     | 32.1% (23.1, 42.7)           | 16.0%                 |
| heterogeneous menstruating-age | Strict Herzog             | 36-month full diary | Windowed Herzog C3 only     | 84                   | 36                     | 42.9% (32.8, 53.5)           | 16.0%                 |
| heterogeneous menstruating-age | Luteal-anchored ovulatory | 36-month full diary | Windowed Herzog thresholds  | 99                   | 31                     | 31.3% (23.0, 41.0)           | 1.0%                  |
| heterogeneous menstruating-age | Strict Herzog             | 36-month full diary | Windowed Herzog thresholds  | 99                   | 42                     | 42.4% (33.2, 52.3)           | 1.0%                  |

**Table 4 caption.** Full-diary and 3-month windowed Herzog results under strict Herzog and luteal-anchored ovulatory phase labeling.

## Table 5. Null study-level prevalence benchmarks

**Why this table is included.** This table maps person-level false positives into apparent prevalence in illustrative studies of 30, 50, and 100 participants. It reports prevalence among all participants, prevalence among classifiable participants only, and the probability of exceeding the 39.1% and 44.2% benchmark values.

**Code to call.**

In [ ]:
study_benchmarks = summary_tables[
    (summary_tables.table_type == "study_level_3month")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.definition.isin([
        "A_windowed_any", "B_minimum_data_C1_or_C2",
        "C_reproducibility_C1_or_C2", "D_nb_regression_C1_or_C2"
    ]))
].copy()
study_benchmarks


| Cohort                         | CE definition                             | Participants per study | Analysis denominator           | Monte Carlo studies | Mean apparent CE prevalence | 2.5th percentile | 97.5th percentile | Probability prevalence at least 39.1% | Probability prevalence at least 44.2% |
| ------------------------------ | ----------------------------------------- | ---------------------- | ------------------------------ | ------------------- | --------------------------- | ---------------- | ----------------- | ------------------------------------- | ------------------------------------- |
| healthy ovulatory              | Windowed Herzog thresholds                | 30.0                   | All participants               | 200                 | 36.9%                       | 20.0%            | 56.7%             | 43.5%                                 | 18.5%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                | 30.0                   | Classifiable participants only | 200                 | 43.9%                       | 26.1%            | 63.0%             | 66.0%                                 | 46.5%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                | 50.0                   | All participants               | 200                 | 36.9%                       | 23.9%            | 48.0%             | 41.0%                                 | 9.0%                                  |
| healthy ovulatory              | Windowed Herzog thresholds                | 50.0                   | Classifiable participants only | 200                 | 44.1%                       | 30.0%            | 59.0%             | 75.5%                                 | 51.5%                                 |
| healthy ovulatory              | Windowed Herzog thresholds                | 100.0                  | All participants               | 200                 | 36.0%                       | 28.0%            | 44.0%             | 16.0%                                 | 1.0%                                  |
| healthy ovulatory              | Windowed Herzog thresholds                | 100.0                  | Classifiable participants only | 200                 | 42.9%                       | 33.7%            | 52.4%             | 79.5%                                 | 40.0%                                 |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 30.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 50.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 100.0                  | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Windowed Herzog C1/C2 with minimum data   | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 30.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 50.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 100.0                  | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| healthy ovulatory              | Negative-binomial regression C1/C2        | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 30.0                   | All participants               | 200                 | 50.2%                       | 33.3%            | 66.8%             | 90.0%                                 | 71.0%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 30.0                   | Classifiable participants only | 200                 | 53.7%                       | 37.0%            | 74.1%             | 94.0%                                 | 86.0%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 50.0                   | All participants               | 200                 | 49.3%                       | 38.0%            | 62.0%             | 95.5%                                 | 71.5%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 50.0                   | Classifiable participants only | 200                 | 53.0%                       | 41.7%            | 65.9%             | 98.5%                                 | 90.0%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 100.0                  | All participants               | 200                 | 50.0%                       | 42.0%            | 59.0%             | 99.0%                                 | 89.0%                                 |
| heterogeneous menstruating-age | Windowed Herzog thresholds                | 100.0                  | Classifiable participants only | 200                 | 53.5%                       | 45.2%            | 62.1%             | 100.0%                                | 99.0%                                 |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 30.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 50.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 100.0                  | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Windowed Herzog C1/C2 with minimum data   | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Cycle reproducibility C1/C2, 6-cycle rule | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 30.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 30.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 50.0                   | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 50.0                   | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 100.0                  | All participants               | 200                 | 0.0%                        | 0.0%             | 0.0%              | 0.0%                                  | 0.0%                                  |
| heterogeneous menstruating-age | Negative-binomial regression C1/C2        | 100.0                  | Classifiable participants only | 0                   | NA                          | NA               | NA                | NA                                    | NA                                    |

**Table 5 caption.** Study-level Monte Carlo summary from null studies using 3-month windows. The interval columns are the 2.5th and 97.5th percentiles of study-level apparent prevalence.

## Table 6. Trial-like conditioned subsets

**Why this table is included.** These subsets answer whether common enrollment restrictions reduce false positives or mainly change the classifiable denominator. The common-classifiable subset supports head-to-head comparisons because every listed core definition is defined on the same windows.

**Code to call.**

In [ ]:
trial_like_subsets = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.window_type == "full")
    & (summary_tables.subset.isin([
        "ge_1_seizure_day_per_month",
        "ge_2_seizures_per_month",
        "strict_23_35_day_cycles_only",
        "common_classifiable_subset",
    ]))
    & (summary_tables.definition.isin([
        "A_windowed_any", "A_windowed_C1_or_C2",
        "B_minimum_data_C1_or_C2",
        "C_reproducibility_C1_or_C2", "D_nb_regression_C1_or_C2"
    ]))
].copy()
trial_like_subsets


| Cohort                         | Analysis denominator             | CE definition                             | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | -------------------------------- | ----------------------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | Common classifiable subset       | Windowed Herzog C1/C2 union               | 9                    | 0                      | 0.0% (0.0, 29.9)             | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Windowed Herzog thresholds                | 9                    | 0                      | 0.0% (0.0, 29.9)             | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Windowed Herzog C1/C2 with minimum data   | 9                    | 0                      | 0.0% (0.0, 29.9)             | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Cycle reproducibility C1/C2, 6-cycle rule | 9                    | 0                      | 0.0% (0.0, 29.9)             | 0.0%                  |
| healthy ovulatory              | Common classifiable subset       | Negative-binomial regression C1/C2        | 9                    | 0                      | 0.0% (0.0, 29.9)             | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Windowed Herzog C1/C2 union               | 73                   | 7                      | 9.6% (4.7, 18.5)             | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Windowed Herzog thresholds                | 73                   | 7                      | 9.6% (4.7, 18.5)             | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Windowed Herzog C1/C2 with minimum data   | 73                   | 7                      | 9.6% (4.7, 18.5)             | 0.0%                  |
| healthy ovulatory              | At least 1 seizure day per month | Cycle reproducibility C1/C2, 6-cycle rule | 9                    | 0                      | 0.0% (0.0, 29.9)             | 87.7%                 |
| healthy ovulatory              | At least 1 seizure day per month | Negative-binomial regression C1/C2        | 73                   | 5                      | 6.8% (3.0, 15.1)             | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Windowed Herzog C1/C2 union               | 54                   | 2                      | 3.7% (1.0, 12.5)             | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Windowed Herzog thresholds                | 54                   | 2                      | 3.7% (1.0, 12.5)             | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Windowed Herzog C1/C2 with minimum data   | 54                   | 2                      | 3.7% (1.0, 12.5)             | 0.0%                  |
| healthy ovulatory              | At least 2 seizures per month    | Cycle reproducibility C1/C2, 6-cycle rule | 9                    | 0                      | 0.0% (0.0, 29.9)             | 83.3%                 |
| healthy ovulatory              | At least 2 seizures per month    | Negative-binomial regression C1/C2        | 54                   | 2                      | 3.7% (1.0, 12.5)             | 0.0%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 union               | 11                   | 1                      | 9.1% (1.6, 37.7)             | 0.0%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Windowed Herzog thresholds                | 11                   | 1                      | 9.1% (1.6, 37.7)             | 0.0%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 with minimum data   | 11                   | 1                      | 9.1% (1.6, 37.7)             | 0.0%                  |
| healthy ovulatory              | Strict 23-35 day cycles only     | Cycle reproducibility C1/C2, 6-cycle rule | 1                    | 0                      | 0.0% (0.0, 79.3)             | 90.9%                 |
| healthy ovulatory              | Strict 23-35 day cycles only     | Negative-binomial regression C1/C2        | 11                   | 0                      | 0.0% (0.0, 25.9)             | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Windowed Herzog C1/C2 union               | 26                   | 3                      | 11.5% (4.0, 29.0)            | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Windowed Herzog thresholds                | 26                   | 11                     | 42.3% (25.5, 61.1)           | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Windowed Herzog C1/C2 with minimum data   | 26                   | 3                      | 11.5% (4.0, 29.0)            | 0.0%                  |
| heterogeneous menstruating-age | Common classifiable subset       | Cycle reproducibility C1/C2, 6-cycle rule | 13                   | 0                      | 0.0% (0.0, 22.8)             | 50.0%                 |
| heterogeneous menstruating-age | Common classifiable subset       | Negative-binomial regression C1/C2        | 26                   | 3                      | 11.5% (4.0, 29.0)            | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Windowed Herzog C1/C2 union               | 78                   | 6                      | 7.7% (3.6, 15.8)             | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Windowed Herzog thresholds                | 78                   | 31                     | 39.7% (29.6, 50.8)           | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Windowed Herzog C1/C2 with minimum data   | 78                   | 6                      | 7.7% (3.6, 15.8)             | 0.0%                  |
| heterogeneous menstruating-age | At least 1 seizure day per month | Cycle reproducibility C1/C2, 6-cycle rule | 13                   | 0                      | 0.0% (0.0, 22.8)             | 83.3%                 |
| heterogeneous menstruating-age | At least 1 seizure day per month | Negative-binomial regression C1/C2        | 78                   | 6                      | 7.7% (3.6, 15.8)             | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Windowed Herzog C1/C2 union               | 63                   | 2                      | 3.2% (0.9, 10.9)             | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Windowed Herzog thresholds                | 63                   | 24                     | 38.1% (27.1, 50.4)           | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Windowed Herzog C1/C2 with minimum data   | 63                   | 2                      | 3.2% (0.9, 10.9)             | 0.0%                  |
| heterogeneous menstruating-age | At least 2 seizures per month    | Cycle reproducibility C1/C2, 6-cycle rule | 13                   | 0                      | 0.0% (0.0, 22.8)             | 79.4%                 |
| heterogeneous menstruating-age | At least 2 seizures per month    | Negative-binomial regression C1/C2        | 63                   | 4                      | 6.3% (2.5, 15.2)             | 0.0%                  |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 union               | 4                    | 0                      | 0.0% (0.0, 49.0)             | 20.0%                 |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Windowed Herzog thresholds                | 4                    | 2                      | 50.0% (15.0, 85.0)           | 20.0%                 |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Windowed Herzog C1/C2 with minimum data   | 4                    | 0                      | 0.0% (0.0, 49.0)             | 20.0%                 |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Cycle reproducibility C1/C2, 6-cycle rule | 1                    | 0                      | 0.0% (0.0, 79.3)             | 80.0%                 |
| heterogeneous menstruating-age | Strict 23-35 day cycles only     | Negative-binomial regression C1/C2        | 4                    | 0                      | 0.0% (0.0, 49.0)             | 20.0%                 |

**Table 6 caption.** Full-diary false-positive rates after applying trial-like eligibility restrictions or a common classifiable denominator. This separates changes in apparent risk from changes in analyzability.

## Table 7. C1/C2/C3 decomposition and C3 exclusion

**Why this table is included.** This table directly addresses whether the heterogeneous-cohort signal is driven by C3 logic. It reports mutually exclusive pattern categories and C1/C2 union comparisons for full-diary windows.

**Code to call.**

In [ ]:
pattern_decomposition = summary_tables[
    (summary_tables.table_type == "pattern_decomposition")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin(["A_windowed", "B_minimum_data", "D_nb_regression"]))
].copy()
pattern_decomposition


| Cohort                         | CE definition   | Pattern category | Classifiable windows | False-positive windows | False-positive rate | Rate among all attempted | Indeterminate windows |
| ------------------------------ | --------------- | ---------------- | -------------------- | ---------------------- | ------------------- | ------------------------ | --------------------- |
| healthy ovulatory              | A_windowed      | C1 only          | 98                   | 6                      | 6.1%                | 6.0%                     | 2.0%                  |
| healthy ovulatory              | A_windowed      | C1+C2            | 98                   | 2                      | 2.0%                | 2.0%                     | 2.0%                  |
| healthy ovulatory              | A_windowed      | C2 only          | 98                   | 8                      | 8.2%                | 8.0%                     | 2.0%                  |
| healthy ovulatory              | A_windowed      | C3 only          | 98                   | 0                      | 0.0%                | 0.0%                     | 2.0%                  |
| healthy ovulatory              | A_windowed      | C3 plus C1/C2    | 98                   | 0                      | 0.0%                | 0.0%                     | 2.0%                  |
| healthy ovulatory              | A_windowed      | none             | 98                   | 82                     | 83.7%               | 82.0%                    | 2.0%                  |
| healthy ovulatory              | B_minimum_data  | C1 only          | 90                   | 5                      | 5.6%                | 5.0%                     | 10.0%                 |
| healthy ovulatory              | B_minimum_data  | C1+C2            | 90                   | 0                      | 0.0%                | 0.0%                     | 10.0%                 |
| healthy ovulatory              | B_minimum_data  | C2 only          | 90                   | 7                      | 7.8%                | 7.0%                     | 10.0%                 |
| healthy ovulatory              | B_minimum_data  | C3 only          | 90                   | 0                      | 0.0%                | 0.0%                     | 10.0%                 |
| healthy ovulatory              | B_minimum_data  | C3 plus C1/C2    | 90                   | 0                      | 0.0%                | 0.0%                     | 10.0%                 |
| healthy ovulatory              | B_minimum_data  | none             | 90                   | 78                     | 86.7%               | 78.0%                    | 10.0%                 |
| healthy ovulatory              | D_nb_regression | C1 only          | 90                   | 2                      | 2.2%                | 2.0%                     | 10.0%                 |
| healthy ovulatory              | D_nb_regression | C1+C2            | 90                   | 0                      | 0.0%                | 0.0%                     | 10.0%                 |
| healthy ovulatory              | D_nb_regression | C2 only          | 90                   | 3                      | 3.3%                | 3.0%                     | 10.0%                 |
| healthy ovulatory              | D_nb_regression | C3 only          | 90                   | 0                      | 0.0%                | 0.0%                     | 10.0%                 |
| healthy ovulatory              | D_nb_regression | C3 plus C1/C2    | 90                   | 0                      | 0.0%                | 0.0%                     | 10.0%                 |
| healthy ovulatory              | D_nb_regression | none             | 90                   | 85                     | 94.4%               | 85.0%                    | 10.0%                 |
| heterogeneous menstruating-age | A_windowed      | C1 only          | 99                   | 4                      | 4.0%                | 4.0%                     | 1.0%                  |
| heterogeneous menstruating-age | A_windowed      | C1+C2            | 99                   | 0                      | 0.0%                | 0.0%                     | 1.0%                  |
| heterogeneous menstruating-age | A_windowed      | C2 only          | 99                   | 2                      | 2.0%                | 2.0%                     | 1.0%                  |
| heterogeneous menstruating-age | A_windowed      | C3 only          | 99                   | 31                     | 31.3%               | 31.0%                    | 1.0%                  |
| heterogeneous menstruating-age | A_windowed      | C3 plus C1/C2    | 99                   | 5                      | 5.1%                | 5.0%                     | 1.0%                  |
| heterogeneous menstruating-age | A_windowed      | none             | 99                   | 57                     | 57.6%               | 57.0%                    | 1.0%                  |
| heterogeneous menstruating-age | B_minimum_data  | C1 only          | 97                   | 3                      | 3.1%                | 3.0%                     | 3.0%                  |
| heterogeneous menstruating-age | B_minimum_data  | C1+C2            | 97                   | 0                      | 0.0%                | 0.0%                     | 3.0%                  |
| heterogeneous menstruating-age | B_minimum_data  | C2 only          | 97                   | 2                      | 2.1%                | 2.0%                     | 3.0%                  |
| heterogeneous menstruating-age | B_minimum_data  | C3 only          | 97                   | 31                     | 32.0%               | 31.0%                    | 3.0%                  |
| heterogeneous menstruating-age | B_minimum_data  | C3 plus C1/C2    | 97                   | 5                      | 5.2%                | 5.0%                     | 3.0%                  |
| heterogeneous menstruating-age | B_minimum_data  | none             | 97                   | 56                     | 57.7%               | 56.0%                    | 3.0%                  |
| heterogeneous menstruating-age | D_nb_regression | C1 only          | 97                   | 3                      | 3.1%                | 3.0%                     | 3.0%                  |
| heterogeneous menstruating-age | D_nb_regression | C1+C2            | 97                   | 0                      | 0.0%                | 0.0%                     | 3.0%                  |
| heterogeneous menstruating-age | D_nb_regression | C2 only          | 97                   | 4                      | 4.1%                | 4.0%                     | 3.0%                  |
| heterogeneous menstruating-age | D_nb_regression | C3 only          | 97                   | 0                      | 0.0%                | 0.0%                     | 3.0%                  |
| heterogeneous menstruating-age | D_nb_regression | C3 plus C1/C2    | 97                   | 0                      | 0.0%                | 0.0%                     | 3.0%                  |
| heterogeneous menstruating-age | D_nb_regression | none             | 97                   | 90                     | 92.8%               | 90.0%                    | 3.0%                  |

**Table 7 caption.** Full-diary pattern decomposition and C3-exclusion sensitivity. C3 is evaluated only when ILP logic is applicable.

## Table 8. Negative-binomial dispersion sensitivity

**Why this table is included.** This table separates the full-diary stabilized-dispersion regression comparator from a full-window, window-only dispersion sensitivity.

**Code to call.**

In [ ]:
nb_dispersion_sensitivity = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type == "full")
    & (summary_tables.definition.isin([
        "D_nb_regression_C1_or_C2", "D_nb_regression_window_alpha_C1_or_C2"
    ]))
].copy()
nb_dispersion_sensitivity


| Cohort                         | Observation window  | CE definition                                              | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows |
| ------------------------------ | ------------------- | ---------------------------------------------------------- | -------------------- | ---------------------- | ---------------------------- | --------------------- |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression C1/C2                         | 90                   | 5                      | 5.6% (2.4, 12.4)             | 10.0%                 |
| healthy ovulatory              | 36-month full diary | Negative-binomial regression C1/C2, window-only dispersion | 90                   | 5                      | 5.6% (2.4, 12.4)             | 10.0%                 |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression C1/C2                         | 97                   | 7                      | 7.2% (3.5, 14.2)             | 3.0%                  |
| heterogeneous menstruating-age | 36-month full diary | Negative-binomial regression C1/C2, window-only dispersion | 97                   | 7                      | 7.2% (3.5, 14.2)             | 3.0%                  |

**Table 8 caption.** Negative-binomial apparent classification rates in full-diary windows using full-diary stabilized alpha and window-only alpha. Both use the same M/O model and Holm family.

## Table 9. Seizure-burden and cycle-regularity strata

**Why this table is included.** The requested strata diagnose where false positives concentrate. Seizure-frequency strata use observed full-diary seizure-days per month, and window-seizure-day strata use total seizure days within each analyzed window.

**Code to call.**

In [ ]:
strata_rows = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.definition.isin(["A_windowed_any", "A_windowed_C1_or_C2", "B_minimum_data_C1_or_C2", "D_nb_regression_C1_or_C2"]))
    & (
        summary_tables.subset.astype(str).str.startswith("seizure_frequency:")
        | summary_tables.subset.astype(str).str.startswith("cycle_regularity:")
        | summary_tables.subset.astype(str).str.startswith("window_seizure_days_")
    )
].copy()
strata_rows


| Cohort                         | Stratum type               | Stratum                           | CE definition                           | Classifiable windows | False-positive windows | False-positive rate | Indeterminate windows |
| ------------------------------ | -------------------------- | --------------------------------- | --------------------------------------- | -------------------- | ---------------------- | ------------------- | --------------------- |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 66                   | 29                     | 43.9%               | 23.3%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 77                   | 33                     | 42.9%               | 10.5%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 77                   | 25                     | 32.5%               | 10.5%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 79                   | 29                     | 36.7%               | 8.1%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 80                   | 19                     | 23.8%               | 7.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 79                   | 22                     | 27.8%               | 8.1%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 83                   | 16                     | 19.3%               | 3.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 82                   | 18                     | 22.0%               | 4.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 84                   | 14                     | 16.7%               | 2.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 74                   | 35                     | 47.3%               | 14.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 80                   | 31                     | 38.8%               | 7.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 79                   | 18                     | 22.8%               | 8.1%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 84                   | 14                     | 16.7%               | 2.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 66                   | 29                     | 43.9%               | 23.3%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 77                   | 33                     | 42.9%               | 10.5%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 77                   | 25                     | 32.5%               | 10.5%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 79                   | 29                     | 36.7%               | 8.1%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 80                   | 19                     | 23.8%               | 7.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 79                   | 22                     | 27.8%               | 8.1%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 83                   | 16                     | 19.3%               | 3.5%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 82                   | 18                     | 22.0%               | 4.7%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 84                   | 14                     | 16.7%               | 2.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 74                   | 35                     | 47.3%               | 14.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 80                   | 31                     | 38.8%               | 7.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 79                   | 18                     | 22.8%               | 8.1%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 84                   | 14                     | 16.7%               | 2.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 62                   | 19                     | 30.6%               | 27.9%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 69                   | 27                     | 39.1%               | 19.8%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 71                   | 18                     | 25.4%               | 17.4%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 72                   | 19                     | 26.4%               | 16.3%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 74                   | 13                     | 17.6%               | 14.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 74                   | 14                     | 18.9%               | 14.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 78                   | 12                     | 15.4%               | 9.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 67                   | 24                     | 35.8%               | 22.1%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 73                   | 16                     | 21.9%               | 15.1%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 78                   | 12                     | 15.4%               | 9.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 78                   | 5                      | 6.4%                | 9.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 1                      | 50.0%               | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 1                      | 50.0%               | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 1                    | 0                      | 0.0%                | 50.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Windowed Herzog C1/C2 with minimum data | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD < 2 days          | Negative-binomial regression C1/C2      | 2                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 7                    | 4                      | 57.1%               | 41.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 10                   | 1                      | 10.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 10                   | 5                      | 50.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 9                    | 3                      | 33.3%               | 25.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 10                   | 2                      | 20.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 10                   | 3                      | 30.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 11                   | 2                      | 18.2%               | 8.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 11                   | 2                      | 18.2%               | 8.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 12                   | 2                      | 16.7%               | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 10                   | 4                      | 40.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 11                   | 6                      | 54.5%               | 8.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 10                   | 4                      | 40.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 12                   | 2                      | 16.7%               | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 7                    | 4                      | 57.1%               | 41.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 10                   | 1                      | 10.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 10                   | 5                      | 50.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 9                    | 3                      | 33.3%               | 25.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 10                   | 2                      | 20.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 10                   | 3                      | 30.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 11                   | 2                      | 18.2%               | 8.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 11                   | 2                      | 18.2%               | 8.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 12                   | 2                      | 16.7%               | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 10                   | 4                      | 40.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 11                   | 6                      | 54.5%               | 8.3%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 10                   | 4                      | 40.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 12                   | 2                      | 16.7%               | 0.0%                  |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 7                    | 2                      | 28.6%               | 41.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 9                    | 3                      | 33.3%               | 25.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 8                    | 1                      | 12.5%               | 33.3%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 10                   | 3                      | 30.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 10                   | 2                      | 20.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 10                   | 1                      | 10.0%               | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 10                   | 0                      | 0.0%                | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 8                    | 5                      | 62.5%               | 33.3%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 9                    | 3                      | 33.3%               | 25.0%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 10                   | 0                      | 0.0%                | 16.7%                 |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 10                   | 0                      | 0.0%                | 16.7%                 |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 52                   | 23                     | 44.2%               | 8.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 56                   | 26                     | 46.4%               | 1.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 56                   | 20                     | 35.7%               | 1.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 57                   | 26                     | 45.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 57                   | 12                     | 21.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 57                   | 18                     | 31.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 57                   | 10                     | 17.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 57                   | 13                     | 22.8%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 57                   | 7                      | 12.3%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 55                   | 26                     | 47.3%               | 3.5%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 57                   | 22                     | 38.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 57                   | 14                     | 24.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 57                   | 7                      | 12.3%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 52                   | 23                     | 44.2%               | 8.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 56                   | 26                     | 46.4%               | 1.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 56                   | 20                     | 35.7%               | 1.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 57                   | 26                     | 45.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 57                   | 12                     | 21.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 57                   | 18                     | 31.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 57                   | 10                     | 17.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 57                   | 13                     | 22.8%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 57                   | 7                      | 12.3%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 55                   | 26                     | 47.3%               | 3.5%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 57                   | 22                     | 38.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 57                   | 14                     | 24.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 57                   | 7                      | 12.3%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 53                   | 19                     | 35.8%               | 7.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 56                   | 26                     | 46.4%               | 1.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 57                   | 12                     | 21.1%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 57                   | 18                     | 31.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 57                   | 10                     | 17.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 57                   | 13                     | 22.8%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 57                   | 7                      | 12.3%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 56                   | 21                     | 37.5%               | 1.8%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 57                   | 14                     | 24.6%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 57                   | 7                      | 12.3%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 57                   | 5                      | 8.8%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 15                   | 7                      | 46.7%               | 6.2%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 2                      | 12.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 2                      | 12.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 2                      | 12.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 3                      | 18.8%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 1                      | 6.2%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 1                      | 6.2%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 6                      | 37.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 4                      | 25.0%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 2                      | 12.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 16                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 15                   | 7                      | 46.7%               | 6.2%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 2                      | 12.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 2                      | 12.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 2                      | 12.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 3                      | 18.8%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 1                      | 6.2%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 1                      | 6.2%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 6                      | 37.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 4                      | 25.0%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 2                      | 12.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 16                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 16                   | 2                      | 12.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 16                   | 2                      | 12.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 16                   | 3                      | 18.8%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 16                   | 1                      | 6.2%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 16                   | 1                      | 6.2%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 16                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 16                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 16                   | 4                      | 25.0%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 16                   | 2                      | 12.5%               | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 16                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 16                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 6                    | 3                      | 50.0%               | 77.8%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 17                   | 7                      | 41.2%               | 37.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 17                   | 8                      | 47.1%               | 37.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 17                   | 4                      | 23.5%               | 37.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 19                   | 6                      | 31.6%               | 29.6%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 18                   | 6                      | 33.3%               | 33.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 23                   | 7                      | 30.4%               | 14.8%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 22                   | 7                      | 31.8%               | 18.5%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 25                   | 9                      | 36.0%               | 7.4%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 15                   | 7                      | 46.7%               | 44.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 20                   | 11                     | 55.0%               | 25.9%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 18                   | 6                      | 33.3%               | 33.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 25                   | 9                      | 36.0%               | 7.4%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 6                    | 3                      | 50.0%               | 77.8%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 17                   | 7                      | 41.2%               | 37.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 17                   | 8                      | 47.1%               | 37.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 17                   | 4                      | 23.5%               | 37.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 19                   | 6                      | 31.6%               | 29.6%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 18                   | 6                      | 33.3%               | 33.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 23                   | 7                      | 30.4%               | 14.8%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 22                   | 7                      | 31.8%               | 18.5%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 25                   | 9                      | 36.0%               | 7.4%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 15                   | 7                      | 46.7%               | 44.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 20                   | 11                     | 55.0%               | 25.9%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 18                   | 6                      | 33.3%               | 33.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 25                   | 9                      | 36.0%               | 7.4%                  |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 1                    | 0                      | 0.0%                | 96.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 8                    | 2                      | 25.0%               | 70.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 8                    | 4                      | 50.0%               | 70.4%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 11                   | 3                      | 27.3%               | 59.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 13                   | 4                      | 30.8%               | 51.9%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 13                   | 2                      | 15.4%               | 51.9%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 17                   | 5                      | 29.4%               | 37.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 5                    | 4                      | 80.0%               | 81.5%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 11                   | 3                      | 27.3%               | 59.3%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 17                   | 5                      | 29.4%               | 37.0%                 |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 17                   | 0                      | 0.0%                | 37.0%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 43                   | 19                     | 44.2%               | 38.6%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 23                   | 11                     | 47.8%               | 32.4%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 19                   | 9                      | 47.4%               | 36.7%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 10                   | 2                      | 20.0%               | 50.0%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 11                   | 2                      | 18.2%               | 42.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 7                    | 3                      | 42.9%               | 56.2%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 10                   | 3                      | 30.0%               | 28.6%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 9                    | 5                      | 55.6%               | 35.7%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 8                    | 4                      | 50.0%               | 20.0%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 25                   | 15                     | 60.0%               | 35.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 16                   | 8                      | 50.0%               | 30.4%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 7                    | 3                      | 42.9%               | 56.2%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 8                    | 4                      | 50.0%               | 20.0%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 43                   | 19                     | 44.2%               | 38.6%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 23                   | 11                     | 47.8%               | 32.4%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 19                   | 9                      | 47.4%               | 36.7%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 10                   | 2                      | 20.0%               | 50.0%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 11                   | 2                      | 18.2%               | 42.1%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 7                    | 3                      | 42.9%               | 56.2%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 10                   | 3                      | 30.0%               | 28.6%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 9                    | 5                      | 55.6%               | 35.7%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 8                    | 4                      | 50.0%               | 20.0%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 25                   | 15                     | 60.0%               | 35.9%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 16                   | 8                      | 50.0%               | 30.4%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 7                    | 3                      | 42.9%               | 56.2%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 8                    | 4                      | 50.0%               | 20.0%                 |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 26                   | 13                     | 50.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 21                   | 12                     | 57.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 17                   | 7                      | 41.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 12                   | 5                      | 41.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 6                    | 4                      | 66.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 8                    | 2                      | 25.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 4                    | 1                      | 25.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 4                    | 3                      | 75.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 19                   | 6                      | 31.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 13                   | 9                      | 69.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 9                    | 3                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 4                    | 3                      | 75.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 26                   | 13                     | 50.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 21                   | 12                     | 57.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 17                   | 7                      | 41.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 12                   | 5                      | 41.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 6                    | 4                      | 66.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 8                    | 2                      | 25.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 4                    | 1                      | 25.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 4                    | 3                      | 75.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 19                   | 6                      | 31.6%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 13                   | 9                      | 69.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 9                    | 3                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 4                    | 3                      | 75.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 17                   | 7                      | 41.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 12                   | 5                      | 41.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 6                    | 4                      | 66.7%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 8                    | 2                      | 25.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4                    | 1                      | 25.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4                    | 3                      | 75.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 13                   | 9                      | 69.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 9                    | 3                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 4                    | 3                      | 75.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 4                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 4                    | 1                      | 25.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 34                   | 12                     | 35.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 31                   | 8                      | 25.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 26                   | 14                     | 53.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 22                   | 6                      | 27.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 11                   | 5                      | 45.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 11                   | 3                      | 27.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 9                    | 2                      | 22.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 3                    | 1                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 33                   | 14                     | 42.4%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 25                   | 8                      | 32.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 15                   | 5                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 3                    | 1                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 4                    | 1                      | 25.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 34                   | 12                     | 35.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 31                   | 8                      | 25.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 26                   | 14                     | 53.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 22                   | 6                      | 27.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 11                   | 5                      | 45.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 11                   | 3                      | 27.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 9                    | 2                      | 22.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 3                    | 1                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 33                   | 14                     | 42.4%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 25                   | 8                      | 32.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 15                   | 5                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 3                    | 1                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 31                   | 8                      | 25.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 26                   | 14                     | 53.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 22                   | 6                      | 27.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11                   | 5                      | 45.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11                   | 3                      | 27.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 9                    | 2                      | 22.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 3                    | 1                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 25                   | 8                      | 32.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 15                   | 5                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 3                    | 1                      | 33.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 3                    | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 11                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 22                   | 6                      | 27.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 42                   | 11                     | 26.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 53                   | 9                      | 17.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 65                   | 15                     | 23.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 71                   | 11                     | 15.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 77                   | 13                     | 16.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 83                   | 8                      | 9.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 9                    | 4                      | 44.4%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 39                   | 12                     | 30.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 60                   | 11                     | 18.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 83                   | 8                      | 9.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 11                   | 0                      | 0.0%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 22                   | 6                      | 27.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 42                   | 11                     | 26.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 53                   | 9                      | 17.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 65                   | 15                     | 23.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 71                   | 11                     | 15.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 77                   | 13                     | 16.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 83                   | 8                      | 9.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 9                    | 4                      | 44.4%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 39                   | 12                     | 30.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 60                   | 11                     | 18.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 83                   | 8                      | 9.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 22                   | 6                      | 27.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 42                   | 11                     | 26.2%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 53                   | 9                      | 17.0%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 65                   | 15                     | 23.1%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 71                   | 11                     | 15.5%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 77                   | 13                     | 16.9%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 83                   | 8                      | 9.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 39                   | 12                     | 30.8%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 60                   | 11                     | 18.3%               | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 83                   | 8                      | 9.6%                | 0.0%                  |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| healthy ovulatory              | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 83                   | 5                      | 6.0%                | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 36                   | 17                     | 47.2%               | 18.2%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 41                   | 18                     | 43.9%               | 6.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 42                   | 20                     | 47.6%               | 4.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 42                   | 12                     | 28.6%               | 4.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 42                   | 12                     | 28.6%               | 4.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 43                   | 13                     | 30.2%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 43                   | 10                     | 23.3%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 43                   | 3                      | 7.0%                | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 43                   | 3                      | 7.0%                | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 41                   | 15                     | 36.6%               | 6.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 43                   | 10                     | 23.3%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 43                   | 12                     | 27.9%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 union             | 43                   | 3                      | 7.0%                | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 36                   | 17                     | 47.2%               | 18.2%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 41                   | 18                     | 43.9%               | 6.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 42                   | 23                     | 54.8%               | 4.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 42                   | 17                     | 40.5%               | 4.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 42                   | 19                     | 45.2%               | 4.5%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 43                   | 22                     | 51.2%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 43                   | 20                     | 46.5%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 43                   | 15                     | 34.9%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 43                   | 20                     | 46.5%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 41                   | 15                     | 36.6%               | 6.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 43                   | 13                     | 30.2%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 43                   | 17                     | 39.5%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog thresholds              | 43                   | 20                     | 46.5%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 35                   | 17                     | 48.6%               | 20.5%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 39                   | 10                     | 25.6%               | 11.4%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 41                   | 11                     | 26.8%               | 6.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 41                   | 11                     | 26.8%               | 6.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 43                   | 10                     | 23.3%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 43                   | 3                      | 7.0%                | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 43                   | 3                      | 7.0%                | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 38                   | 8                      | 21.1%               | 13.6%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 43                   | 12                     | 27.9%               | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Windowed Herzog C1/C2 with minimum data | 43                   | 3                      | 7.0%                | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD 2 to <4 days      | Negative-binomial regression C1/C2      | 43                   | 3                      | 7.0%                | 2.3%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 46                   | 21                     | 45.7%               | 17.9%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 52                   | 20                     | 38.5%               | 7.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 52                   | 21                     | 40.4%               | 7.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 54                   | 18                     | 33.3%               | 3.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 55                   | 19                     | 34.5%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 55                   | 17                     | 30.9%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 56                   | 16                     | 28.6%               | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 56                   | 14                     | 25.0%               | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 56                   | 8                      | 14.3%               | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 53                   | 23                     | 43.4%               | 5.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 55                   | 18                     | 32.7%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 56                   | 20                     | 35.7%               | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 union             | 56                   | 8                      | 14.3%               | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 46                   | 22                     | 47.8%               | 17.9%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 52                   | 25                     | 48.1%               | 7.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 52                   | 29                     | 55.8%               | 7.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 54                   | 29                     | 53.7%               | 3.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 55                   | 30                     | 54.5%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 55                   | 28                     | 50.9%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 56                   | 27                     | 48.2%               | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 56                   | 26                     | 46.4%               | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 56                   | 22                     | 39.3%               | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 53                   | 31                     | 58.5%               | 5.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 55                   | 32                     | 58.2%               | 1.8%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 56                   | 27                     | 48.2%               | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog thresholds              | 56                   | 22                     | 39.3%               | 0.0%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 45                   | 16                     | 35.6%               | 19.6%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 46                   | 16                     | 34.8%               | 17.9%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 52                   | 17                     | 32.7%               | 7.1%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 53                   | 15                     | 28.3%               | 5.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 53                   | 14                     | 26.4%               | 5.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 54                   | 13                     | 24.1%               | 3.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 54                   | 7                      | 13.0%               | 3.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 4                    | 1                      | 25.0%               | 92.9%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 49                   | 16                     | 32.7%               | 12.5%                 |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 53                   | 17                     | 32.1%               | 5.4%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Windowed Herzog C1/C2 with minimum data | 54                   | 7                      | 13.0%               | 3.6%                  |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Cycle-regularity stratum   | Cycle length SD at least 4 days   | Negative-binomial regression C1/C2      | 54                   | 4                      | 7.4%                | 3.6%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 57                   | 26                     | 45.6%               | 3.4%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 26                     | 44.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 30                     | 50.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 18                     | 30.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 16                     | 27.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 21                     | 35.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 12                     | 20.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 10                     | 16.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 6                      | 10.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 20                     | 33.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 18                     | 30.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 18                     | 30.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 union             | 59                   | 6                      | 10.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 57                   | 26                     | 45.6%               | 3.4%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 27                     | 45.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 36                     | 61.0%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 27                     | 45.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 28                     | 47.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 31                     | 52.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 26                     | 44.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 24                     | 40.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 25                     | 42.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 25                     | 42.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 28                     | 47.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 24                     | 40.7%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog thresholds              | 59                   | 25                     | 42.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 56                   | 29                     | 51.8%               | 5.1%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 58                   | 18                     | 31.0%               | 1.7%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 59                   | 16                     | 27.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 59                   | 21                     | 35.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 59                   | 12                     | 20.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 59                   | 10                     | 16.9%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 59                   | 6                      | 10.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 2                    | 1                      | 50.0%               | 96.6%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 59                   | 18                     | 30.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 59                   | 18                     | 30.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Windowed Herzog C1/C2 with minimum data | 59                   | 6                      | 10.2%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 1 to <4 seizure days per month    | Negative-binomial regression C1/C2      | 59                   | 5                      | 8.5%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 18                   | 6                      | 33.3%               | 5.3%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 5                      | 26.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 2                      | 10.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 5                      | 26.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 5                      | 26.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 1                      | 5.3%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 3                      | 15.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 7                      | 36.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 3                      | 15.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 2                      | 10.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 union             | 19                   | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 18                   | 7                      | 38.9%               | 5.3%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 8                      | 42.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 7                      | 36.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 9                      | 47.4%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 7                      | 36.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 6                      | 31.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 6                      | 31.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 5                      | 26.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 6                      | 31.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 8                      | 42.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 8                      | 42.1%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 6                      | 31.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog thresholds              | 19                   | 6                      | 31.6%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 19                   | 2                      | 10.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 19                   | 5                      | 26.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 19                   | 5                      | 26.3%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 19                   | 1                      | 5.3%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 19                   | 3                      | 15.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 19                   | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 19                   | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 1                    | 0                      | 0.0%                | 94.7%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 19                   | 3                      | 15.8%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 19                   | 2                      | 10.5%               | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Windowed Herzog C1/C2 with minimum data | 19                   | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | 4 to <10 seizure days per month   | Negative-binomial regression C1/C2      | 19                   | 1                      | 5.3%                | 0.0%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 7                    | 6                      | 85.7%               | 68.2%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 15                   | 7                      | 46.7%               | 31.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 16                   | 9                      | 56.2%               | 27.3%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 18                   | 7                      | 38.9%               | 18.2%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 19                   | 10                     | 52.6%               | 13.6%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 20                   | 8                      | 40.0%               | 9.1%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 21                   | 11                     | 52.4%               | 4.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 21                   | 7                      | 33.3%               | 4.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 21                   | 5                      | 23.8%               | 4.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 16                   | 11                     | 68.8%               | 27.3%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 20                   | 7                      | 35.0%               | 9.1%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 21                   | 12                     | 57.1%               | 4.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 union             | 21                   | 5                      | 23.8%               | 4.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 7                    | 6                      | 85.7%               | 68.2%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 15                   | 8                      | 53.3%               | 31.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 16                   | 9                      | 56.2%               | 27.3%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 18                   | 10                     | 55.6%               | 18.2%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 19                   | 14                     | 73.7%               | 13.6%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 20                   | 13                     | 65.0%               | 9.1%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 21                   | 15                     | 71.4%               | 4.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 21                   | 12                     | 57.1%               | 4.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 21                   | 11                     | 52.4%               | 4.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 16                   | 13                     | 81.2%               | 27.3%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 20                   | 9                      | 45.0%               | 9.1%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 21                   | 14                     | 66.7%               | 4.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog thresholds              | 21                   | 11                     | 52.4%               | 4.5%                  |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 5                    | 2                      | 40.0%               | 77.3%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 8                    | 3                      | 37.5%               | 63.6%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 15                   | 7                      | 46.7%               | 31.8%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 16                   | 4                      | 25.0%               | 27.3%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 18                   | 9                      | 50.0%               | 18.2%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 19                   | 6                      | 31.6%               | 13.6%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 19                   | 4                      | 21.1%               | 13.6%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 1                    | 0                      | 0.0%                | 95.5%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 9                    | 3                      | 33.3%               | 59.1%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 18                   | 9                      | 50.0%               | 18.2%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Windowed Herzog C1/C2 with minimum data | 19                   | 4                      | 21.1%               | 13.6%                 |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Seizure-frequency stratum  | Less than 1 seizure day per month | Negative-binomial regression C1/C2      | 19                   | 1                      | 5.3%                | 13.6%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 56                   | 27                     | 48.2%               | 24.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 17                   | 9                      | 52.9%               | 29.2%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 14                   | 8                      | 57.1%               | 30.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 11                   | 4                      | 36.4%               | 26.7%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 4                    | 3                      | 75.0%               | 42.9%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 4                    | 4                      | 100.0%              | 33.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 3                    | 2                      | 66.7%               | 25.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 2                    | 1                      | 50.0%               | 33.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 2                    | 1                      | 50.0%               | 33.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 22                   | 13                     | 59.1%               | 21.4%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 11                   | 4                      | 36.4%               | 15.4%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 3                    | 3                      | 100.0%              | 25.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 union             | 2                    | 1                      | 50.0%               | 33.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 56                   | 28                     | 50.0%               | 24.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 17                   | 10                     | 58.8%               | 29.2%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 14                   | 8                      | 57.1%               | 30.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 11                   | 6                      | 54.5%               | 26.7%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 4                    | 3                      | 75.0%               | 42.9%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 4                    | 4                      | 100.0%              | 33.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 3                    | 2                      | 66.7%               | 25.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 2                    | 2                      | 100.0%              | 33.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 2                    | 1                      | 50.0%               | 33.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 22                   | 15                     | 68.2%               | 21.4%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 11                   | 5                      | 45.5%               | 15.4%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 3                    | 3                      | 100.0%              | 25.0%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog thresholds              | 2                    | 1                      | 50.0%               | 33.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 0-3 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 24                   | 11                     | 45.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 31                   | 14                     | 45.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 24                   | 10                     | 41.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 15                   | 6                      | 40.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 15                   | 6                      | 40.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 8                    | 2                      | 25.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 6                    | 2                      | 33.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 1                    | 1                      | 100.0%              | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 1                    | 1                      | 100.0%              | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 27                   | 11                     | 40.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 13                   | 6                      | 46.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 11                   | 5                      | 45.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 union             | 1                    | 1                      | 100.0%              | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 24                   | 11                     | 45.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 31                   | 14                     | 45.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 24                   | 12                     | 50.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 15                   | 9                      | 60.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 15                   | 10                     | 66.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 8                    | 5                      | 62.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 6                    | 3                      | 50.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 1                    | 1                      | 100.0%              | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 1                    | 1                      | 100.0%              | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 27                   | 15                     | 55.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 13                   | 7                      | 53.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 11                   | 6                      | 54.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog thresholds              | 1                    | 1                      | 100.0%              | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 24                   | 10                     | 41.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 15                   | 6                      | 40.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 15                   | 6                      | 40.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 8                    | 2                      | 25.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 6                    | 2                      | 33.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 1                    | 1                      | 100.0%              | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 1                    | 1                      | 100.0%              | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 2                    | 1                      | 50.0%               | 92.6%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 13                   | 6                      | 46.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 11                   | 5                      | 45.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Windowed Herzog C1/C2 with minimum data | 1                    | 1                      | 100.0%              | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 4-7 seizure days                  | Negative-binomial regression C1/C2      | 1                    | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 2                    | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 37                   | 13                     | 35.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 32                   | 17                     | 53.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 31                   | 11                     | 35.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 16                   | 6                      | 37.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 13                   | 5                      | 38.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 11                   | 7                      | 63.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 12                   | 3                      | 25.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 4                    | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 39                   | 12                     | 30.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 24                   | 5                      | 20.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 12                   | 7                      | 58.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 union             | 4                    | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 2                    | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 37                   | 16                     | 43.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 32                   | 22                     | 68.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 31                   | 14                     | 45.2%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 16                   | 11                     | 68.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 13                   | 7                      | 53.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 11                   | 10                     | 90.9%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 12                   | 4                      | 33.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 4                    | 1                      | 25.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 39                   | 13                     | 33.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 24                   | 11                     | 45.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 12                   | 7                      | 58.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog thresholds              | 4                    | 1                      | 25.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 32                   | 17                     | 53.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 31                   | 11                     | 35.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 16                   | 6                      | 37.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 13                   | 5                      | 38.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 11                   | 7                      | 63.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 12                   | 3                      | 25.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 4                    | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 1                    | 0                      | 0.0%                | 97.4%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 24                   | 5                      | 20.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 12                   | 7                      | 58.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Windowed Herzog C1/C2 with minimum data | 4                    | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | 8-15 seizure days                 | Negative-binomial regression C1/C2      | 4                    | 0                      | 0.0%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 8                    | 2                      | 25.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 24                   | 6                      | 25.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 39                   | 9                      | 23.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 62                   | 16                     | 25.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 73                   | 19                     | 26.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 79                   | 15                     | 19.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 84                   | 12                     | 14.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 92                   | 9                      | 9.8%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 6                    | 2                      | 33.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 50                   | 13                     | 26.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 73                   | 17                     | 23.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 union             | 92                   | 9                      | 9.8%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 8                    | 3                      | 37.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 24                   | 10                     | 41.7%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 39                   | 17                     | 43.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 62                   | 25                     | 40.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 73                   | 34                     | 46.6%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 79                   | 32                     | 40.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 84                   | 34                     | 40.5%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 92                   | 39                     | 42.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 6                    | 3                      | 50.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 50                   | 22                     | 44.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 73                   | 28                     | 38.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog thresholds              | 92                   | 39                     | 42.4%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 24                   | 6                      | 25.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 39                   | 9                      | 23.1%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 62                   | 16                     | 25.8%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 73                   | 19                     | 26.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 79                   | 15                     | 19.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 84                   | 12                     | 14.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 92                   | 9                      | 9.8%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 1                    | 0                      | 0.0%                | 83.3%                 |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 50                   | 13                     | 26.0%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 73                   | 17                     | 23.3%               | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Windowed Herzog C1/C2 with minimum data | 92                   | 9                      | 9.8%                | 0.0%                  |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 0                    | 0                      | NA                  | 100.0%                |
| heterogeneous menstruating-age | Window seizure-day stratum | >=16 seizure days                 | Negative-binomial regression C1/C2      | 92                   | 7                      | 7.6%                | 0.0%                  |

**Table 9 caption.** Apparent classification rates by observed seizure burden and cycle regularity strata. These strata identify where null positives are concentrated.

## Table 10. Assumption-based historical definitions

**Why this table is included.** Historical rules are exploratory operationalizations rather than literal replications, so they are flagged separately. This table keeps them out of the core endpoint table while still showing their null false-positive behavior.

**Code to call.**

In [ ]:
historical_rows = summary_tables[
    (summary_tables.table_type == "window_false_positive")
    & (summary_tables.phase_mode == "strict_herzog")
    & (summary_tables.subset == "all")
    & (summary_tables.window_type.isin(["calendar", "full"]))
    & (summary_tables.definition.isin([
        "H1_newmark_penry_any", "H1_newmark_penry_66_7_any",
        "H2_duncan1993_any", "H3_herzog1997_twofold_any",
        "H4_reddy2007_any_phase2x_any"
    ]))
].copy()
historical_rows


| Cohort                         | Observation window  | CE definition                        | Classifiable windows | False-positive windows | False-positive rate (95% CI) | Indeterminate windows | Assumption-based historical rule |
| ------------------------------ | ------------------- | ------------------------------------ | -------------------- | ---------------------- | ---------------------------- | --------------------- | -------------------------------- |
| healthy ovulatory              | 3 months            | Newmark-Penry two-thirds sensitivity | 89                   | 3                      | 3.4% (1.2, 9.4)              | 11.0%                 | Yes                              |
| healthy ovulatory              | 3 months            | Newmark-Penry perimenstrual rule     | 89                   | 7                      | 7.9% (3.9, 15.4)             | 11.0%                 | Yes                              |
| healthy ovulatory              | 3 months            | Duncan 1993 ten-day rule             | 89                   | 5                      | 5.6% (2.4, 12.5)             | 11.0%                 | Yes                              |
| healthy ovulatory              | 3 months            | Herzog 1997 twofold rule             | 89                   | 32                     | 36.0% (26.8, 46.3)           | 11.0%                 | Yes                              |
| healthy ovulatory              | 3 months            | Reddy 2007 any-phase twofold rule    | 89                   | 64                     | 71.9% (61.8, 80.2)           | 11.0%                 | Yes                              |
| healthy ovulatory              | 36-month full diary | Newmark-Penry two-thirds sensitivity | 98                   | 0                      | 0.0% (0.0, 3.8)              | 2.0%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Newmark-Penry perimenstrual rule     | 98                   | 1                      | 1.0% (0.2, 5.6)              | 2.0%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Duncan 1993 ten-day rule             | 98                   | 0                      | 0.0% (0.0, 3.8)              | 2.0%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Herzog 1997 twofold rule             | 98                   | 15                     | 15.3% (9.5, 23.7)            | 2.0%                  | Yes                              |
| healthy ovulatory              | 36-month full diary | Reddy 2007 any-phase twofold rule    | 98                   | 17                     | 17.3% (11.1, 26.0)           | 2.0%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Newmark-Penry two-thirds sensitivity | 93                   | 4                      | 4.3% (1.7, 10.5)             | 7.0%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Newmark-Penry perimenstrual rule     | 93                   | 9                      | 9.7% (5.2, 17.4)             | 7.0%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Duncan 1993 ten-day rule             | 93                   | 7                      | 7.5% (3.7, 14.7)             | 7.0%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Herzog 1997 twofold rule             | 93                   | 43                     | 46.2% (36.5, 56.3)           | 7.0%                  | Yes                              |
| heterogeneous menstruating-age | 3 months            | Reddy 2007 any-phase twofold rule    | 93                   | 67                     | 72.0% (62.2, 80.1)           | 7.0%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Newmark-Penry two-thirds sensitivity | 99                   | 1                      | 1.0% (0.2, 5.5)              | 1.0%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Newmark-Penry perimenstrual rule     | 99                   | 2                      | 2.0% (0.6, 7.1)              | 1.0%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Duncan 1993 ten-day rule             | 99                   | 1                      | 1.0% (0.2, 5.5)              | 1.0%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Herzog 1997 twofold rule             | 99                   | 36                     | 36.4% (27.6, 46.2)           | 1.0%                  | Yes                              |
| heterogeneous menstruating-age | 36-month full diary | Reddy 2007 any-phase twofold rule    | 99                   | 11                     | 11.1% (6.3, 18.8)            | 1.0%                  | Yes                              |

**Table 10 caption.** Apparent classification rates for exploratory historical definitions. These rows are deliberately labeled as assumption-based and should not be interpreted as literal historical replications.

## Table 11. Output manifest

**Why this table is included.** The manifest is machine-readable provenance: it lists every analysis artifact, size, checksum, and the assumptions that were not directly derivable from simulator outputs.

**Code to call.**

In [ ]:
manifest_files = pd.DataFrame(manifest["files"])
manifest_files.assign(size_mb=manifest_files["bytes"] / 1_000_000)[["path", "size_mb", "sha256"]]


| Output file                                                     | Size, MB | SHA-256 prefix      |
| --------------------------------------------------------------- | -------- | ------------------- |
| outputs_bench_200/audit_daily_sample.parquet                    | 0.039    | 68af35e603a27dbc... |
| outputs_bench_200/fig1_false_positive_by_window.pdf             | 0.008    | c73b3b8cf615f950... |
| outputs_bench_200/fig1_false_positive_by_window.png             | 0.016    | 6474caa03df9d898... |
| outputs_bench_200/fig1_false_positive_by_window.svg             | 0.013    | e450f73b87ca5713... |
| outputs_bench_200/fig2_pattern_decomposition.pdf                | 0.017    | 3663500594351fc1... |
| outputs_bench_200/fig2_pattern_decomposition.png                | 0.077    | 2a04e648373f555f... |
| outputs_bench_200/fig2_pattern_decomposition.svg                | 0.056    | 3b1b8ba87070a87a... |
| outputs_bench_200/fig3_study_prevalence_distribution_3month.pdf | 0.018    | 893bfe1562c8da65... |
| outputs_bench_200/fig3_study_prevalence_distribution_3month.png | 0.099    | 6546c228482af842... |
| outputs_bench_200/fig3_study_prevalence_distribution_3month.svg | 0.068    | 76183516d31b880f... |
| outputs_bench_200/fig4_historical_vs_core_definitions.pdf       | 0.017    | d947ad5295aa4e0f... |
| outputs_bench_200/fig4_historical_vs_core_definitions.png       | 0.078    | f8386f7b13b708ce... |
| outputs_bench_200/fig4_historical_vs_core_definitions.svg       | 0.049    | 00b107e6360ec480... |
| outputs_bench_200/fig4_indeterminate_vs_fpr_frontier.pdf        | 0.025    | c16bcdb595dd8509... |
| outputs_bench_200/fig4_indeterminate_vs_fpr_frontier.png        | 0.175    | a86e32f911c0775d... |
| outputs_bench_200/fig4_indeterminate_vs_fpr_frontier.svg        | 0.099    | a9823434458738fe... |
| outputs_bench_200/fig5_qc_null_cycle_day_profile.pdf            | 0.017    | f596d65505f9fa67... |
| outputs_bench_200/fig5_qc_null_cycle_day_profile.png            | 0.105    | 1e4fb10be302e84d... |
| outputs_bench_200/fig5_qc_null_cycle_day_profile.svg            | 0.073    | d85381d5a0eca76b... |
| outputs_bench_200/participant_summary.parquet                   | 0.035    | 49eb37af680b772f... |
| outputs_bench_200/progress.json                                 | 0.000    | 87bd92253edbbeb5... |
| outputs_bench_200/study_level_3month.parquet                    | 0.072    | 7305be0df8bca969... |
| outputs_bench_200/study_level_3month_n30.parquet                | 0.072    | 7305be0df8bca969... |
| outputs_bench_200/summary_tables.csv                            | 3.923    | 0b9527d8f754c883... |
| outputs_bench_200/window_results.parquet                        | 0.357    | e72ac8e87b33b7bd... |

**Table 11 caption.** Machine-readable output provenance. The checksum prefix is included to support reproducibility checks without making the table unnecessarily wide.

## Publication-ready figures

Each figure is written as PNG for notebook viewing and PDF/SVG for publication workflows. Fractional outcomes are displayed on a 0-100% percentage scale. The code cell below is the function call that regenerates the figure set from the populated output tables.

In [ ]:
from paper1_null_ce.core.plots import write_all_figures

# Regenerate PNG and PDF figures from current populated output tables.
write_all_figures(OUTPUT_DIR, summary_tables, study_level, pd.read_parquet(OUTPUT_DIR / "audit_daily_sample.parquet"))


### Figure 1. False-positive rate by window

Calendar-month false-positive rate for practical monitoring durations, split by cohort.

PDF companion: [PDF version](../outputs_bench_200/fig1_false_positive_by_window.pdf)

![Figure 1. False-positive rate by window](../outputs_bench_200/fig1_false_positive_by_window.png)

**Figure caption.** Apparent classification rates are shown as percentages among classifiable participant windows for random calendar windows. The dashed reference line marks 5%.

### Figure 2. C-pattern decomposition

Mutually exclusive C1/C2/C3 pattern categories for full-diary windows.

PDF companion: [PDF version](../outputs_bench_200/fig2_pattern_decomposition.pdf)

![Figure 2. C-pattern decomposition](../outputs_bench_200/fig2_pattern_decomposition.png)

**Figure caption.** Bars show the share of all attempted full-diary windows in each pattern category, including indeterminate windows, under strict Herzog phase labeling.

### Figure 3. Study prevalence distribution

Study-level Monte Carlo distribution for 3-month null studies with n=30, n=50, and n=100.

PDF companion: [PDF version](../outputs_bench_200/fig3_study_prevalence_distribution_3month.pdf)

![Figure 3. Study prevalence distribution](../outputs_bench_200/fig3_study_prevalence_distribution_3month.png)

**Figure caption.** Each curve summarizes simulated studies using random 3-month windows and the windowed Herzog threshold definition. The y-axis is the proportion of simulated studies; vertical reference lines mark benchmark apparent CE prevalence values.

### Figure 4. Indeterminate versus false-positive frontier

Tradeoff between rejecting underspecified windows and the false-positive rate among classifiable windows.

PDF companion: [PDF version](../outputs_bench_200/fig4_indeterminate_vs_fpr_frontier.pdf)

![Figure 4. Indeterminate versus false-positive frontier](../outputs_bench_200/fig4_indeterminate_vs_fpr_frontier.png)

**Figure caption.** Each point is a definition-by-window-by-cohort result. Points farther right have more indeterminate windows; points higher on the plot have more false positives among windows that remained classifiable.

### Appendix Figure. Historical versus core definitions

Assumption-based historical rules compared with core protocol definitions.

PDF companion: [PDF version](../outputs_bench_200/fig4_historical_vs_core_definitions.pdf)

![Appendix Figure. Historical versus core definitions](../outputs_bench_200/fig4_historical_vs_core_definitions.png)

**Figure caption.** The historical definitions are exploratory operationalizations and are plotted next to the core definitions only to show their null false-positive behavior under the same 3-month window setting.

### Appendix QC Figure. Null cycle-day seizure profile

Quality-control cycle-day seizure profile in the daily audit sample.

PDF companion: [PDF version](../outputs_bench_200/fig5_qc_null_cycle_day_profile.pdf)

![Appendix QC Figure. Null cycle-day seizure profile](../outputs_bench_200/fig5_qc_null_cycle_day_profile.png)

**Figure caption.** The audit sample contains 1% of participant daily rows. Lines show average daily seizure frequency by observed menstrual cycle day with approximate Poisson error bars.

## Interpretation notes

- The notebook is populated from the current files in `outputs/`. If those files were produced by smoke mode, the numerical values are smoke-test values, not the final 100,000-participant estimates.
- Full-study values are produced by running `run_paper1_null_ce.py --config config.yaml --full`, then rebuilding this notebook.
- Exact Herzog 2004 results are intentionally present only for 3-complete-cycle windows.
- Historical definitions are assumption-based operationalizations and should be kept separate from core endpoints.
- The manifest assumptions are part of the analysis record:

  - Definition D uses a participant-full-diary method-of-moments negative-binomial alpha recorded in d_alpha; Poisson robust fallback is recorded in d_reason when statsmodels NB fitting fails. Definition D_window_alpha re-estimates alpha from the analyzed window as a non-oracle sensitivity.
  - Healthy ovulatory cohort used hormone_cycler build_patient_profile/render_cycle with ovulation_probability set to 1.0 because simulate_diary does not expose a public force-ovulation knob.
  - Historical definitions H1-H4 are assumption-based operationalizations and are flagged in summary outputs.
  - Study-level Monte Carlo samples each selected participant from a deterministic pool of precomputed random valid 3-month windows to avoid retaining all daily diaries in memory.
  - The hormone simulator exposes medical-factor knobs but no natural prevalence sampler; heterogeneous menstruating-age medical factors were sampled from config.yaml rates.